# Data Pipeline Notebook

This notebook combines:
1. Raw data collection (Fundamentals, Stock Prices, Options)
2. Point-in-time adjustment
3. Feature engineering
4. Integrity and coverage validation


## 1) Environment And Imports


In [23]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Ensure project-root imports when running notebook from notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "team_t").exists():
    TEAM_ROOT = CWD / "team_t"
elif CWD.name == "team_t":
    TEAM_ROOT = CWD
elif CWD.name == "notebooks" and CWD.parent.name == "team_t":
    TEAM_ROOT = CWD.parent
else:
    TEAM_ROOT = next((p for p in [CWD, *CWD.parents] if p.name == "team_t"), None)

if TEAM_ROOT is None:
    raise FileNotFoundError("Could not resolve team_t directory from current working directory")

if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from src.data_utils import (
    configure_api_from_env,
    fetch_zacks_table,
    load_prices_csv_required,
    prepare_fundamentals_with_availability,
    asof_join_point_in_time,
    validate_point_in_time_panel,
    connect_wrds,
    pull_optionmetrics_calls_atm_dataset,
)
from src.universe_selection import (
    build_rebalance_calendar,
    build_annual_candidate_table,
    finalize_annual_universe_with_options,
    expand_annual_membership_to_daily,
    attach_universe_flags,
)
from src.feature_engineering import (
    add_split_adjusted_intraday_prices,
    add_fundamental_change_features,
    add_price_liquidity_features,
    add_staged_features,
    get_stage_feature_columns,
    get_cross_section_rank_feature_columns,
    compute_price_to_book,
    compute_rolling_beta_vs_spy,
    winsorize_cross_sectional,
    rank_cross_sectional,
    zscore_cross_sectional,
)


## 2) Parameters And Output Paths


In [ ]:
# Full-history defaults for forward-walk workflows.
PRICE_START = pd.Timestamp("1900-01-01")
PRICE_END = pd.Timestamp.today().normalize()
LAG_DAYS = 45

# Dynamic PIT universe settings.
UNIVERSE_START_YEAR = 2006
UNIVERSE_END_YEAR = 2024
REBALANCE_ANCHOR_MONTH = 5
REBALANCE_ANCHOR_DAY = 15
UNIVERSE_SIZE = 30
UNIVERSE_BUFFER_SIZE = 15
MISSINGNESS_MAX = 0.40
ADDV_MIN = 20_000_000
MIN_PRICE = 5.0
OPTIONS_REBAL_WINDOW_DAYS = 5

# Pull window for universe construction and model sample.
SCREEN_START = pd.Timestamp(f"{UNIVERSE_START_YEAR - 1}-01-01")
SCREEN_END = pd.Timestamp(f"{UNIVERSE_END_YEAR}-12-31")

BETA_WINDOW = 252
BETA_MIN_OBS = 126
FEATURE_STAGE_MAX = 5

OPT_START = None
OPT_END = None

# Cache controls for faster reruns.
# Set any of these to True when you need a fresh pull/recompute.
USE_CACHE = True
REFRESH_FUNDAMENTALS = True
REFRESH_PRICES = True
REFRESH_OPTIONS = False   # keep this False (no 2-hour options repull)
REFRESH_FEATURE_PANEL = True


PRICES_CSV = TEAM_ROOT / "data" / "PRICES.csv"
OUTPUT_DIR = TEAM_ROOT / "data" / "lstm_draft" / "processed"
OPTIONS_DIR = TEAM_ROOT / "data" / "options"
for path in [OUTPUT_DIR, OPTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

UNIVERSE_CSV = OUTPUT_DIR / "lstm_dynamic_universe_top30.csv"
UNIVERSE_OPTIONS_SCOPE_CSV = OUTPUT_DIR / "lstm_dynamic_universe_options_scope.csv"
UNIVERSE_REBALANCE_CSV = OUTPUT_DIR / "universe_rebalance_schedule.csv"
UNIVERSE_CANDIDATES_PARQUET = OUTPUT_DIR / "universe_candidates_annual.parquet"
UNIVERSE_ANNUAL_PARQUET = OUTPUT_DIR / "universe_annual_final.parquet"
UNIVERSE_DAILY_PARQUET = OUTPUT_DIR / "universe_daily_membership.parquet"

# Raw exports (full-history canonical)
PRICES_ADJ_PARQUET_FULL = OUTPUT_DIR / "prices_adjusted_full_history.parquet"
FUNDAMENTALS_PARQUET_FULL = OUTPUT_DIR / "fundamentals_with_availability_full_history.parquet"
MKTV_PARQUET_FULL = OUTPUT_DIR / "mktv_quarterly_full_history.parquet"
BETA_PARQUET_FULL = OUTPUT_DIR / "beta_252d_full_history.parquet"
OPTIONS_PARQUET_FULL = OPTIONS_DIR / "optionmetrics_calls_atm_20_60d_dynamic_universe_full_history.parquet"

# Raw exports (legacy names)
PRICES_ADJ_PARQUET = OUTPUT_DIR / "prices_adjusted_2006_2013.parquet"
FUNDAMENTALS_PARQUET = OUTPUT_DIR / "fundamentals_with_availability_2006_2013.parquet"
BETA_PARQUET = OUTPUT_DIR / "beta_252d_2006_2013.parquet"
OPTIONS_PARQUET = OPTIONS_DIR / "optionmetrics_calls_atm_20_60d_2006_2013.parquet"

# Feature panel exports
FEATURE_PANEL_PARQUET = OUTPUT_DIR / "feature_panel_pit_normalized_full_history.parquet"
FEATURE_PANEL_CSV = OUTPUT_DIR / "feature_panel_pit_normalized_full_history.csv"

# Backward-compatible panel names consumed in older flows.
PANEL_PARQUET_FULL = OUTPUT_DIR / "lstm_feature_panel_full_history.parquet"
PANEL_CSV_FULL = OUTPUT_DIR / "lstm_feature_panel_full_history.csv"
PANEL_PARQUET = OUTPUT_DIR / "lstm_feature_panel_2006_2013.parquet"
PANEL_CSV = OUTPUT_DIR / "lstm_feature_panel_2006_2013.csv"


## 3) Credentials Setup


In [25]:
env_candidates = [
    TEAM_ROOT / ".env",
    TEAM_ROOT.parent / ".env",
    Path.cwd() / ".env",
]
configure_api_from_env(env_candidates)


## Fundamentals (quarterly)

Used to construct the signal.

Examples:
- ROE
- profit_margin
- debt_equity
- EPS
- book_value_per_share

These come from Zacks or similar sources.


In [26]:
mktv_cols = ["ticker", "per_end_date", "per_type", "mkt_val"]
fr_cols = [
    "ticker", "per_end_date", "per_type",
    "tot_debt_tot_equity", "ret_equity", "profit_margin", "book_val_per_share",
]
fc_cols = ["ticker", "per_end_date", "per_type", "diluted_net_eps"]

date_lo = SCREEN_START.date().isoformat()
date_hi = SCREEN_END.date().isoformat()


def _fetch_by_ticker_chunks(
    table_code: str,
    columns: list[str],
    tickers: list[str],
    date_lo: str,
    date_hi: str,
    chunk_size: int = 250,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for start in range(0, len(tickers), chunk_size):
        chunk = tickers[start : start + chunk_size]
        if not chunk:
            continue
        part = fetch_zacks_table(
            table_code,
            columns,
            filters={"ticker": {"in": chunk}, "per_end_date": {"between": (date_lo, date_hi)}},
            paginate=True,
        )
        if not part.empty:
            frames.append(part)

    if not frames:
        return pd.DataFrame(columns=columns)

    return pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)


if (
    USE_CACHE
    and not REFRESH_FUNDAMENTALS
    and FUNDAMENTALS_PARQUET_FULL.exists()
    and MKTV_PARQUET_FULL.exists()
):
    fundamentals = pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
    mktv = pd.read_parquet(MKTV_PARQUET_FULL)

    if "per_end_date" in fundamentals.columns:
        fundamentals["per_end_date"] = pd.to_datetime(fundamentals["per_end_date"], errors="coerce")
    if "feature_available_date" in fundamentals.columns:
        fundamentals["feature_available_date"] = pd.to_datetime(
            fundamentals["feature_available_date"], errors="coerce"
        )

    mktv["per_end_date"] = pd.to_datetime(mktv["per_end_date"], errors="coerce")
    mktv["mkt_val"] = pd.to_numeric(mktv["mkt_val"], errors="coerce")

    print(f"[cache hit] fundamentals={FUNDAMENTALS_PARQUET_FULL}")
    print(f"[cache hit] mktv={MKTV_PARQUET_FULL}")
else:
    mktv = fetch_zacks_table(
        "ZACKS/MKTV",
        mktv_cols,
        filters={"per_end_date": {"between": (date_lo, date_hi)}},
        paginate=True,
    )
    mktv["per_end_date"] = pd.to_datetime(mktv["per_end_date"], errors="coerce")
    if "per_type" in mktv.columns:
        mktv = mktv[mktv["per_type"].astype(str).str.upper() == "Q"].copy()

    raw_tickers = sorted(mktv["ticker"].astype(str).dropna().str.strip().unique().tolist())
    if not raw_tickers:
        raise ValueError("MKTV fetch returned no ticker candidates for dynamic universe build")

    print(f"[mktv] rows={len(mktv):,} raw_tickers={len(raw_tickers):,}")
    fr = _fetch_by_ticker_chunks("ZACKS/FR", fr_cols, raw_tickers, date_lo=date_lo, date_hi=date_hi)
    fc = _fetch_by_ticker_chunks("ZACKS/FC", fc_cols, raw_tickers, date_lo=date_lo, date_hi=date_hi)
    fundamentals = prepare_fundamentals_with_availability(fr, fc, lag_days=LAG_DAYS)

candidate_price_tickers = sorted(
    fundamentals["ticker_price"].astype(str).str.upper().str.strip().dropna().unique().tolist()
)
if not candidate_price_tickers:
    raise ValueError("No candidate tickers available after fundamentals preparation")

print(f"[universe candidate pool] tickers={len(candidate_price_tickers):,} sample={candidate_price_tickers[:5]}")
print(f"[fundamentals] rows={len(fundamentals):,} tickers={fundamentals['ticker_price'].nunique():,}")
print(f"[fundamentals] per_end_date min={fundamentals['per_end_date'].min()} max={fundamentals['per_end_date'].max()}")
print(f"[mktv] rows={len(mktv):,} tickers={mktv['ticker'].astype(str).nunique():,}")


[mktv] rows=1,000,000 raw_tickers=13,158
[universe candidate pool] tickers=11,556 sample=['A', 'AA', 'AAAP', 'AABA', 'AAC']
[fundamentals] rows=405,705 tickers=11,556
[fundamentals] per_end_date min=2006-01-31 00:00:00 max=2024-12-31 00:00:00
[mktv] rows=1,000,000 tickers=13,158


## Stock Prices (daily)

Required for:
- return targets
- delta hedge
- realized volatility

Fields:
- date
- ticker
- open
- close
- adj_open
- adj_close
- volume


In [27]:
price_cols = ["ticker", "date", "open", "close", "adj_close", "volume"]
screen_tickers = sorted(set(candidate_price_tickers + ["SPY"]))

use_price_cache = USE_CACHE and not REFRESH_PRICES and PRICES_ADJ_PARQUET_FULL.exists()
if use_price_cache:
    prices_adj = pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
    prices_adj["date"] = pd.to_datetime(prices_adj["date"], errors="coerce")
    prices_adj["ticker"] = prices_adj["ticker"].astype(str).str.upper().str.strip()

    cached_tickers = set(prices_adj["ticker"].dropna().astype(str).unique().tolist())
    missing_required = sorted(set(screen_tickers) - cached_tickers)
    if missing_required:
        print(
            f"[cache note] price cache missing {len(missing_required):,} screen tickers; "
            "using cached prices. Set REFRESH_PRICES=True to rebuild from PRICES.csv."
        )
    else:
        print(f"[cache hit] prices_adj={PRICES_ADJ_PARQUET_FULL}")

if not use_price_cache:
    prices = load_prices_csv_required(
        csv_path=PRICES_CSV,
        tickers=screen_tickers,
        start=SCREEN_START.date().isoformat(),
        end=SCREEN_END.date().isoformat(),
        usecols=price_cols,
    )
    prices_adj = add_split_adjusted_intraday_prices(prices)

prices_adj["date"] = pd.to_datetime(prices_adj["date"], errors="coerce")
prices_adj["ticker"] = prices_adj["ticker"].astype(str).str.upper().str.strip()
prices_adj = prices_adj[(prices_adj["date"] >= SCREEN_START) & (prices_adj["date"] <= SCREEN_END)].copy()
prices_adj = prices_adj.sort_values(["ticker", "date"]).reset_index(drop=True)

if prices_adj.empty:
    raise ValueError("No price rows returned for selected ticker universe and date range.")

print(f"[prices_adj] rows={len(prices_adj):,} tickers={prices_adj['ticker'].nunique():,}")
print(f"[prices_adj] date_min={prices_adj['date'].min()} date_max={prices_adj['date'].max()}")
prices_adj[["date", "ticker", "open", "close", "adj_open", "adj_close", "volume"]].head(3)


[prices_adj] rows=19,233,748 tickers=7,381
[prices_adj] date_min=2005-01-03 00:00:00 date_max=2024-11-05 00:00:00


,date,ticker,open,close,adj_open,adj_close,volume
0,2005-01-03,A,24.10,23.88,15.607669,15.465192,2416900.0
1,2005-01-04,A,23.78,23.25,15.400430,15.057191,2680200.0
2,2005-01-05,A,23.17,23.24,15.005381,15.050714,2788700.0


### Export Raw Fundamentals + Prices


In [28]:
prices_for_universe = prices_adj[prices_adj["ticker"] != "SPY"].copy()

rebalance_schedule = build_rebalance_calendar(
    prices_df=prices_for_universe,
    start_year=UNIVERSE_START_YEAR,
    end_year=UNIVERSE_END_YEAR,
    anchor_month=REBALANCE_ANCHOR_MONTH,
    anchor_day=REBALANCE_ANCHOR_DAY,
)

candidates_annual = build_annual_candidate_table(
    mktv_df=mktv,
    fundamentals_df=fundamentals,
    prices_df=prices_for_universe,
    rebalance_df=rebalance_schedule,
    target_size=UNIVERSE_SIZE,
    buffer_size=UNIVERSE_BUFFER_SIZE,
    missingness_max=MISSINGNESS_MAX,
    addv_min=ADDV_MIN,
    min_price=MIN_PRICE,
)
if candidates_annual.empty:
    raise ValueError("Dynamic universe candidate table is empty; cannot proceed")

options_scope_tickers = sorted(
    candidates_annual.loc[candidates_annual["pre_options_pass"], "ticker"].astype(str).dropna().unique().tolist()
)
if not options_scope_tickers:
    raise ValueError("No pre-options candidate tickers survived filters")
options_scope_df = pd.DataFrame({"ticker": options_scope_tickers})

rebalance_schedule.to_csv(UNIVERSE_REBALANCE_CSV, index=False)
candidates_annual.to_parquet(UNIVERSE_CANDIDATES_PARQUET, index=False)
options_scope_df.to_csv(UNIVERSE_OPTIONS_SCOPE_CSV, index=False)

mktv.to_parquet(MKTV_PARQUET_FULL, index=False)
fundamentals.to_parquet(FUNDAMENTALS_PARQUET_FULL, index=False)
prices_adj.to_parquet(PRICES_ADJ_PARQUET_FULL, index=False)

# Backward-compatible raw filenames
fundamentals.to_parquet(FUNDAMENTALS_PARQUET, index=False)
prices_adj.to_parquet(PRICES_ADJ_PARQUET, index=False)

candidate_year_counts = (
    candidates_annual.loc[candidates_annual["pre_options_pass"]]
    .groupby("year")["ticker"]
    .nunique()
    .rename("pre_options_survivors")
)

print(f"[export] rebalance_schedule={UNIVERSE_REBALANCE_CSV}")
print(f"[export] candidates_annual={UNIVERSE_CANDIDATES_PARQUET}")
print(f"[export] options_scope={UNIVERSE_OPTIONS_SCOPE_CSV}")
print(f"[export] fundamentals_full={FUNDAMENTALS_PARQUET_FULL}")
print(f"[export] prices_adj_full={PRICES_ADJ_PARQUET_FULL}")
print(f"[candidates] pre_options_rows={len(candidates_annual):,} scope_tickers={len(options_scope_tickers):,}")
print(candidate_year_counts.to_string())


[export] rebalance_schedule=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/universe_rebalance_schedule.csv
[export] candidates_annual=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/universe_candidates_annual.parquet
[export] options_scope=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_dynamic_universe_options_scope.csv
[export] fundamentals_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/fundamentals_with_availability_full_history.parquet
[export] prices_adj_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/prices_adjusted_full_history.parquet
[candidates] pre_options_rows=175,649 scope_tickers=1,374
year
2006     309
2007     366
2008     419
2009     397
2010     404
2011     437
2012     457
2013     481
2014     557
2015     611
2016     650
2017     690
2018     757
2019     795
2020     798
2021     926
2022     952
2023     927
2024    1017


## Options Data (daily)

From OptionMetrics IvyDB US.

Used for:
- ATM option prices
- delta
- implied volatility

Example fields:
- date
- ticker
- strike
- expiration
- delta
- mid_price
- implied_vol
- open_interest


In [29]:
opt_start_date = OPT_START if OPT_START is not None else prices_adj["date"].min().date().isoformat()
print(f"[options pull window] start={opt_start_date} end={OPT_END if OPT_END is not None else 'latest available'}")

options_scope_df = pd.read_csv(UNIVERSE_OPTIONS_SCOPE_CSV)
options_scope_tickers = sorted(
    options_scope_df["ticker"].astype(str).str.upper().str.strip().dropna().unique().tolist()
)

use_options_cache = USE_CACHE and not REFRESH_OPTIONS and OPTIONS_PARQUET_FULL.exists()
options_df = None

if use_options_cache:
    cached = pd.read_parquet(OPTIONS_PARQUET_FULL)
    cached["ticker"] = cached["ticker"].astype(str).str.upper().str.strip()
    dte_ok = ("dte" in cached.columns) and cached["dte"].between(30, 60).all()
    cp_ok = ("cp_flag" in cached.columns) and cached["cp_flag"].astype(str).str.upper().eq("C").all()
    if dte_ok and cp_ok:
        options_df = cached
        print(f"[cache hit] options={OPTIONS_PARQUET_FULL}")
        print(f"[cache hit] rows={len(options_df):,} tickers={options_df['ticker'].nunique():,}")
    else:
        print(
            "[cache stale] options cache failed validation "
            f"(dte_ok={dte_ok}, cp_ok={cp_ok}). Re-pulling."
        )

if options_df is None:
    db = connect_wrds()
    try:
        options_df = pull_optionmetrics_calls_atm_dataset(
            db=db,
            universe_path=UNIVERSE_OPTIONS_SCOPE_CSV,
            fallback_universe_path=UNIVERSE_OPTIONS_SCOPE_CSV,
            output_path=OPTIONS_PARQUET_FULL,
            start_date=opt_start_date,
            end_date=OPT_END,
        )
    finally:
        db.close()
        print("[db] WRDS connection closed")

if OPTIONS_PARQUET_FULL.exists() and OPTIONS_PARQUET_FULL != OPTIONS_PARQUET:
    options_df.to_parquet(OPTIONS_PARQUET, index=False)

trading_calendar_df = prices_adj[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
universe_annual = finalize_annual_universe_with_options(
    candidates_df=candidates_annual,
    options_df=options_df,
    trading_calendar_df=trading_calendar_df,
    target_size=UNIVERSE_SIZE,
    window_days=OPTIONS_REBAL_WINDOW_DAYS,
)
if universe_annual.empty:
    raise ValueError("Final annual dynamic universe is empty after options gate")

universe_daily = expand_annual_membership_to_daily(
    annual_universe_df=universe_annual,
    trading_calendar_df=trading_calendar_df,
)
if universe_daily.empty:
    raise ValueError("Daily universe membership expansion produced zero rows")

universe_tickers = sorted(universe_annual["ticker"].astype(str).unique().tolist())
pd.DataFrame({"ticker": universe_tickers}).to_csv(UNIVERSE_CSV, index=False)
universe_annual.to_parquet(UNIVERSE_ANNUAL_PARQUET, index=False)
universe_daily.to_parquet(UNIVERSE_DAILY_PARQUET, index=False)

if (universe_annual.groupby("selection_year")["ticker"].nunique() > UNIVERSE_SIZE).any():
    raise ValueError("Universe year contains more than target size entries")

print(f"[options] rows={len(options_df):,} tickers={options_df['ticker'].nunique():,}")
print(f"[options] output_full={OPTIONS_PARQUET_FULL}")
print(f"[options] output_legacy={OPTIONS_PARQUET}")
print(f"[universe] annual={UNIVERSE_ANNUAL_PARQUET} rows={len(universe_annual):,}")
print(f"[universe] daily={UNIVERSE_DAILY_PARQUET} rows={len(universe_daily):,}")
print(f"[universe] tickers={len(universe_tickers):,} sample={universe_tickers[:5]}")
print(universe_annual.groupby("selection_year")["ticker"].nunique().rename("final_count").to_string())


[options pull window] start=2005-01-03 end=latest available
[cache hit] options=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_dynamic_universe_full_history.parquet
[cache hit] rows=2,514,990 tickers=661
[options] rows=2,514,990 tickers=661
[options] output_full=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_dynamic_universe_full_history.parquet
[options] output_legacy=/Users/assortedsphinx/Desktop/team_t/data/options/optionmetrics_calls_atm_20_60d_2006_2013.parquet
[universe] annual=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/universe_annual_final.parquet rows=570
[universe] daily=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/universe_daily_membership.parquet rows=141,660
[universe] tickers=58 sample=['ABT', 'ACN', 'ADBE', 'AIG', 'AMD']
selection_year
2006    30
2007    30
2008    30
2009    30
2010    30
2011    30
2012    30
2013    30
2014    30
2015    30
2016    30
2017    30
2

## Point-In-Time Adjustment + Feature Engineering

This section:
- aligns fundamentals by availability date using PIT as-of join
- computes change-based and price/liquidity features
- applies cross-sectional winsorization and z-score normalization


In [30]:
expected_stage_features = get_stage_feature_columns(max_stage=FEATURE_STAGE_MAX)
required_dynamic_cols = ["in_universe", "has_option_today", "tradable_today"]

use_feature_cache = (
    USE_CACHE
    and not REFRESH_FEATURE_PANEL
    and FEATURE_PANEL_PARQUET.exists()
    and BETA_PARQUET_FULL.exists()
)

if use_feature_cache:
    panel_out = pd.read_parquet(FEATURE_PANEL_PARQUET)
    beta_df = pd.read_parquet(BETA_PARQUET_FULL)
    panel_out["date"] = pd.to_datetime(panel_out["date"], errors="coerce")

    feature_cols = [col for col in expected_stage_features if col in panel_out.columns]
    missing_stage_cols = [col for col in expected_stage_features if col not in panel_out.columns]
    missing_dynamic_cols = [col for col in required_dynamic_cols if col not in panel_out.columns]

    if missing_stage_cols or missing_dynamic_cols:
        print(
            "[cache stale] feature panel missing required columns: "
            f"staged={missing_stage_cols} dynamic={missing_dynamic_cols}. Recomputing feature panel."
        )
        use_feature_cache = False
    else:
        print(f"[cache hit] feature_panel={FEATURE_PANEL_PARQUET}")
        print(f"[cache hit] feature_stage={FEATURE_STAGE_MAX} feature_count={len(feature_cols)}")

if not use_feature_cache:
    if "universe_annual" not in globals() and UNIVERSE_ANNUAL_PARQUET.exists():
        universe_annual = pd.read_parquet(UNIVERSE_ANNUAL_PARQUET)
    if "universe_daily" not in globals() and UNIVERSE_DAILY_PARQUET.exists():
        universe_daily = pd.read_parquet(UNIVERSE_DAILY_PARQUET)

    selected_tickers = sorted(universe_annual["ticker"].astype(str).str.upper().str.strip().unique().tolist())
    if not selected_tickers:
        raise ValueError("No selected tickers in final annual universe")

    fund_pti = fundamentals.copy()
    fund_pti["ticker"] = fund_pti["ticker_price"].astype(str).str.upper().str.strip()
    fund_pti = fund_pti[fund_pti["ticker"].isin(selected_tickers)].copy()

    prices_model = prices_adj[prices_adj["ticker"].isin(set(selected_tickers + ["SPY"]))].copy()
    prices_model["ticker"] = prices_model["ticker"].astype(str).str.upper().str.strip()

    panel_prices = prices_model[prices_model["ticker"].isin(selected_tickers)].copy()
    panel = asof_join_point_in_time(
        prices_df=panel_prices,
        fundamentals_df=fund_pti,
        on_date_col="date",
        by_ticker_col="ticker",
    )
    validate_point_in_time_panel(panel)

    panel_feat = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
    panel_feat = attach_universe_flags(
        panel_df=panel_feat,
        daily_membership_df=universe_daily,
        options_df=options_df,
    )
    panel_feat = compute_price_to_book(panel_feat)

    # Stage 1 baseline features (unchanged formulas).
    panel_feat = add_fundamental_change_features(panel_feat, ticker_col="ticker", date_col="date")
    panel_feat = add_price_liquidity_features(
        panel_feat,
        ticker_col="ticker",
        date_col="date",
        price_col="adj_close",
        volume_col="volume",
        volume_window=20,
        min_volume_obs=5,
    )

    beta_df = compute_rolling_beta_vs_spy(
        prices_df=prices_model[["ticker", "date", "adj_close"]].copy(),
        window=BETA_WINDOW,
        min_obs=BETA_MIN_OBS,
    )
    panel_feat = panel_feat.merge(beta_df[["ticker", "date", "beta_252d"]], on=["ticker", "date"], how="left")
    panel_feat = panel_feat.rename(columns={"beta_252d": "rolling_beta"})

    # Stages 2-5 derived features are cumulative and use availability-time alignment.
    panel_feat = add_staged_features(
        panel_df=panel_feat,
        max_stage=FEATURE_STAGE_MAX,
        ticker_col="ticker",
        date_col="date",
        feature_available_col="feature_available_date",
        price_col="adj_close",
        reaction_clip=(-0.05, 0.05),
        vol_epsilon=1e-8,
    )

    feature_cols = [col for col in expected_stage_features if col in panel_feat.columns]
    missing_stage_cols = [col for col in expected_stage_features if col not in panel_feat.columns]
    if missing_stage_cols:
        raise KeyError(f"Missing staged feature columns after build: {missing_stage_cols}")

    # Restrict normalization and modeling features to active universe rows.
    if "in_universe" not in panel_feat.columns:
        raise KeyError("Expected 'in_universe' flag before normalization")
    inactive_mask = ~panel_feat["in_universe"].fillna(False)
    panel_feat.loc[inactive_mask, feature_cols] = np.nan

    # One-pass normalization after all staged raw features are generated.
    rank_feature_cols = get_cross_section_rank_feature_columns(feature_cols)
    raw_scale_feature_cols = [col for col in feature_cols if col not in rank_feature_cols]

    panel_feat = winsorize_cross_sectional(
        panel_df=panel_feat,
        feature_cols=raw_scale_feature_cols,
        date_col="date",
        lower_q=0.01,
        upper_q=0.99,
    )
    panel_feat = rank_cross_sectional(
        panel_df=panel_feat,
        feature_cols=rank_feature_cols,
        date_col="date",
        suffix="_rank",
        center=True,
        universe_col="in_universe",
    )
    rank_source_overrides = {col: f"{col}_rank" for col in rank_feature_cols}
    panel_feat = zscore_cross_sectional(
        panel_df=panel_feat,
        feature_cols=feature_cols,
        date_col="date",
        suffix="_z",
        source_overrides=rank_source_overrides,
    )

    # Keep backward-compatible next-day target alongside engineered features.
    panel_feat["adj_close"] = pd.to_numeric(panel_feat["adj_close"], errors="coerce")
    panel_feat.loc[panel_feat["adj_close"] <= 0, "adj_close"] = np.nan
    panel_feat["target_next_log_ret"] = panel_feat.groupby("ticker")["adj_close"].transform(
        lambda s: np.log(s.shift(-1)) - np.log(s)
    )

    panel_out = panel_feat.sort_values(["ticker", "date"]).reset_index(drop=True)

panel_out = panel_out.sort_values(["ticker", "date"]).reset_index(drop=True)
duplicates = int(panel_out.duplicated(["ticker", "date"]).sum())
assert duplicates == 0, f"Found {duplicates} duplicate (ticker, date) rows in panel_out"

# Export engineered feature panel.
panel_out.to_parquet(FEATURE_PANEL_PARQUET, index=False)
panel_out.to_csv(FEATURE_PANEL_CSV, index=False)

# Backward-compatible panel exports.
panel_out.to_parquet(PANEL_PARQUET_FULL, index=False)
panel_out.to_csv(PANEL_CSV_FULL, index=False)
panel_out.to_parquet(PANEL_PARQUET, index=False)
panel_out.to_csv(PANEL_CSV, index=False)

# Export beta artifacts retained from previous raw-data workflow.
beta_df.to_parquet(BETA_PARQUET_FULL, index=False)
beta_df.to_parquet(BETA_PARQUET, index=False)

print(f"[panel] rows={len(panel_out):,} tickers={panel_out['ticker'].nunique():,}")
print(f"[panel] date_min={panel_out['date'].min()} date_max={panel_out['date'].max()}")
print(f"[panel] feature_stage={FEATURE_STAGE_MAX}")
print(f"[panel] feature_cols={feature_cols}")
print(f"[panel] in_universe_rows={int(panel_out['in_universe'].sum()):,}")
print(f"[panel] tradable_rows={int(panel_out['tradable_today'].sum()):,}")
print(f"[export] feature_panel_parquet={FEATURE_PANEL_PARQUET}")
print(f"[export] feature_panel_csv={FEATURE_PANEL_CSV}")
print(f"[export] beta_full={BETA_PARQUET_FULL}")


[panel] rows=287,840 tickers=58
[panel] date_min=2005-01-03 00:00:00 date_max=2024-11-05 00:00:00
[panel] feature_stage=5
[panel] feature_cols=['roe_change', 'margin_change', 'leverage_change', 'eps_growth', 'book_value_growth', 'log_return', 'rolling_beta', 'volume_ratio', 'roe_change_accel', 'margin_change_accel', 'eps_growth_surprise', 'reaction_speed', 'vol_regime_ratio_20_60']
[panel] in_universe_rows=139,530
[panel] tradable_rows=136,037
[export] feature_panel_parquet=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/feature_panel_pit_normalized_full_history.parquet
[export] feature_panel_csv=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/feature_panel_pit_normalized_full_history.csv
[export] beta_full=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/beta_252d_full_history.parquet


## 4) Integrity And Summary Of Data


In [31]:
def _date_min_max(df: pd.DataFrame, date_col: str) -> tuple[str | None, str | None]:
    if df is None or df.empty or date_col not in df.columns:
        return None, None
    d = pd.to_datetime(df[date_col], errors="coerce")
    return (
        str(d.min().date()) if d.notna().any() else None,
        str(d.max().date()) if d.notna().any() else None,
    )

_fund_df = fundamentals if "fundamentals" in globals() else pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
_prices_df = prices_adj if "prices_adj" in globals() else pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
_opt_df = options_df if "options_df" in globals() else pd.read_parquet(OPTIONS_PARQUET_FULL)
_panel_df = panel_out if "panel_out" in globals() else pd.read_parquet(FEATURE_PANEL_PARQUET)

coverage = pd.DataFrame(
    [
        {
            "dataset": "Fundamentals (quarterly)",
            "date_field": "per_end_date",
            "date_min": _date_min_max(_fund_df, "per_end_date")[0],
            "date_max": _date_min_max(_fund_df, "per_end_date")[1],
            "rows": int(len(_fund_df)),
            "tickers": int(_fund_df["ticker_price"].nunique()) if "ticker_price" in _fund_df.columns else None,
        },
        {
            "dataset": "Fundamentals availability (PIT)",
            "date_field": "feature_available_date",
            "date_min": _date_min_max(_fund_df, "feature_available_date")[0],
            "date_max": _date_min_max(_fund_df, "feature_available_date")[1],
            "rows": int(len(_fund_df)),
            "tickers": int(_fund_df["ticker_price"].nunique()) if "ticker_price" in _fund_df.columns else None,
        },
        {
            "dataset": "Stock Prices (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_prices_df, "date")[0],
            "date_max": _date_min_max(_prices_df, "date")[1],
            "rows": int(len(_prices_df)),
            "tickers": int(_prices_df["ticker"].nunique()) if "ticker" in _prices_df.columns else None,
        },
        {
            "dataset": "Options Data (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_opt_df, "date")[0],
            "date_max": _date_min_max(_opt_df, "date")[1],
            "rows": int(len(_opt_df)),
            "tickers": int(_opt_df["ticker"].nunique()) if "ticker" in _opt_df.columns else None,
        },
        {
            "dataset": "PIT Feature Panel (daily)",
            "date_field": "date",
            "date_min": _date_min_max(_panel_df, "date")[0],
            "date_max": _date_min_max(_panel_df, "date")[1],
            "rows": int(len(_panel_df)),
            "tickers": int(_panel_df["ticker"].nunique()) if "ticker" in _panel_df.columns else None,
        },
    ]
)
coverage


,dataset,date_field,date_min,date_max,rows,tickers
0,Fundamentals (quarterly),per_end_date,2006-01-31,2024-12-31,405705,11556
1,Fundamentals availability (PIT),feature_available_date,2006-03-17,2025-02-14,405705,11556
2,Stock Prices (daily),date,2005-01-03,2024-11-05,19233748,7381
3,Options Data (daily),date,2005-01-03,2025-08-29,2514990,661
4,PIT Feature Panel (daily),date,2005-01-03,2024-11-05,287840,58


In [32]:
# Validate required raw-data blocks and expected fields.
_fund_df = fundamentals if "fundamentals" in globals() else pd.read_parquet(FUNDAMENTALS_PARQUET_FULL)
_prices_df = prices_adj if "prices_adj" in globals() else pd.read_parquet(PRICES_ADJ_PARQUET_FULL)
_opt_df = options_df if "options_df" in globals() else pd.read_parquet(OPTIONS_PARQUET_FULL)
_panel_df = panel_out if "panel_out" in globals() else pd.read_parquet(FEATURE_PANEL_PARQUET)

required_fund = {
    "ROE": ["ret_equity", "roe"],
    "profit_margin": ["profit_margin"],
    "debt_equity": ["tot_debt_tot_equity", "debt_equity"],
    "EPS": ["diluted_net_eps", "eps", "eps_basic"],
    "book_value_per_share": ["book_val_per_share", "book_value_per_share"],
}
required_prices = {
    "date": ["date"],
    "ticker": ["ticker"],
    "open": ["open"],
    "close": ["close"],
    "adj_open": ["adj_open"],
    "adj_close": ["adj_close", "adj_close_intraday"],
    "volume": ["volume"],
}
required_options = {
    "date": ["date"],
    "ticker": ["ticker"],
    "strike": ["strike_price", "strike"],
    "expiration": ["exdate", "expiration"],
    "delta": ["delta"],
    "mid_price": ["mid_price"],
    "implied_vol": ["implied_vol", "implied_volatility", "impl_volatility"],
    "open_interest": ["open_interest"],
}
required_panel = {
    "date": ["date"],
    "ticker": ["ticker"],
}
required_panel.update({col: [col] for col in get_stage_feature_columns(max_stage=FEATURE_STAGE_MAX)})

def _present(requirements: dict[str, list[str]], cols: list[str]) -> dict[str, str | None]:
    colset = set(cols)
    out: dict[str, str | None] = {}
    for logical, aliases in requirements.items():
        out[logical] = next((a for a in aliases if a in colset), None)
    return out

def _status(mapping: dict[str, str | None]) -> str:
    return "PASS" if all(v is not None for v in mapping.values()) else "FAIL"

rows = [
    {
        "dataset": "Fundamentals (quarterly)",
        "status": _status(_present(required_fund, list(_fund_df.columns))),
    },
    {
        "dataset": "Stock Prices (daily)",
        "status": _status(_present(required_prices, list(_prices_df.columns))),
    },
    {
        "dataset": "Options Data (daily)",
        "status": _status(_present(required_options, list(_opt_df.columns))),
    },
    {
        "dataset": "PIT Feature Panel (daily)",
        "status": _status(_present(required_panel, list(_panel_df.columns))),
    },
]
validation_table = pd.DataFrame(rows)

if (validation_table["status"] == "FAIL").any():
    raise AssertionError("Validation failed:\n" + validation_table.to_string(index=False))

validation_table


,dataset,status
0,Fundamentals (quarterly),PASS
1,Stock Prices (daily),PASS
2,Options Data (daily),PASS
3,PIT Feature Panel (daily),PASS


In [33]:
# Final artifact integrity check.
required_files = [
    UNIVERSE_CSV,
    FUNDAMENTALS_PARQUET_FULL,
    PRICES_ADJ_PARQUET_FULL,
    BETA_PARQUET_FULL,
    OPTIONS_PARQUET_FULL,
    FEATURE_PANEL_PARQUET,
    FEATURE_PANEL_CSV,
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required exported files: {missing_files}")

opt_chk = pd.read_parquet(OPTIONS_PARQUET_FULL)
required_option_cols = {
    "date", "ticker", "secid", "cp_flag", "strike_price", "exdate", "dte",
    "best_bid", "best_offer", "mid_price", "implied_vol", "delta",
    "underlying_price", "moneyness", "option_mid_return",
}
missing_option_cols = sorted(required_option_cols - set(opt_chk.columns))
if missing_option_cols:
    raise KeyError(f"Options missing columns: {missing_option_cols}")

checks = {}
if not opt_chk.empty:
    checks["calls_only"] = bool(opt_chk["cp_flag"].astype(str).str.upper().eq("C").all())
    checks["dte_30_60"] = bool(opt_chk["dte"].between(30, 60).all())
    checks["positive_bid"] = bool((opt_chk["best_bid"] > 0).all())
    checks["positive_offer"] = bool((opt_chk["best_offer"] > 0).all())
    checks["moneyness_band"] = bool(opt_chk["moneyness"].between(0.95, 1.05).all())
    checks["one_contract_per_ticker_date"] = bool(not opt_chk.duplicated(["ticker", "date"]).any())
    if "open_interest" in opt_chk.columns and opt_chk["open_interest"].notna().any():
        checks["positive_open_interest"] = bool((opt_chk["open_interest"] > 0).all())

integrity_manifest = {
    "fundamentals_rows": int(len(pd.read_parquet(FUNDAMENTALS_PARQUET_FULL))),
    "prices_rows": int(len(pd.read_parquet(PRICES_ADJ_PARQUET_FULL))),
    "options_rows": int(len(opt_chk)),
    "feature_panel_rows": int(len(pd.read_parquet(FEATURE_PANEL_PARQUET))),
    "checks": checks,
}
integrity_manifest


{'fundamentals_rows': 405705,
 'prices_rows': 19233748,
 'options_rows': 2514990,
 'feature_panel_rows': 287840,
 'checks': {'calls_only': True,
  'dte_30_60': True,
  'positive_bid': True,
  'positive_offer': True,
  'moneyness_band': True,
  'one_contract_per_ticker_date': True,
  'positive_open_interest': True}}